# 🗑️ SwachhCity - Garbage Detection Model Training

This notebook trains a YOLOv8 model for garbage detection.

**Run this on Google Colab for free GPU!**

## Steps:
1. Install dependencies
2. Download/prepare dataset
3. Train YOLOv8 model
4. Export model
5. Test inference

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics opencv-python pillow numpy torch torchvision
!pip install roboflow  # For downloading datasets

In [ ]:
# Step 2: Import libraries
import os
import shutil
from pathlib import Path
from ultralytics import YOLO
import torch

# Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 3: Download garbage dataset from Roboflow (pre-labeled)
# This is a public garbage detection dataset

from roboflow import Roboflow

# Using a public garbage detection dataset
# You can replace this with your own dataset
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")  # Get free key at roboflow.com

# Alternative: Download TACO dataset directly
!git clone https://github.com/pedropro/TACO.git

# Or use this public garbage dataset (no API key needed)
!wget https://github.com/ultralytics/assets/releases/download/v0.0.0/garbage-detection.zip
!unzip -q garbage-detection.zip -d ./datasets/

In [ ]:
# Step 3 Alternative: Create dataset from TrashNet
# This creates a binary classification dataset

import urllib.request
import zipfile

# Download TrashNet
print("Downloading TrashNet dataset...")
url = "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"
urllib.request.urlretrieve(url, "trashnet.zip")

# Extract
with zipfile.ZipFile("trashnet.zip", 'r') as zip_ref:
    zip_ref.extractall("./trashnet")

print("Dataset downloaded!")

In [ ]:
# Step 4: Prepare dataset for YOLOv8 classification
import os
import shutil
import random
from pathlib import Path

# Create YOLO dataset structure
dataset_path = Path("./garbage_dataset")
(dataset_path / "train" / "garbage").mkdir(parents=True, exist_ok=True)
(dataset_path / "train" / "no_garbage").mkdir(parents=True, exist_ok=True)
(dataset_path / "val" / "garbage").mkdir(parents=True, exist_ok=True)
(dataset_path / "val" / "no_garbage").mkdir(parents=True, exist_ok=True)

# Copy TrashNet images to garbage folder
trashnet_path = Path("./trashnet/dataset-resized")
garbage_categories = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

all_images = []
for cat in garbage_categories:
    cat_path = trashnet_path / cat
    if cat_path.exists():
        all_images.extend(list(cat_path.glob('*.jpg')))

random.shuffle(all_images)
split = int(len(all_images) * 0.8)

# Copy to train/val
for i, img in enumerate(all_images[:split]):
    shutil.copy(img, dataset_path / "train" / "garbage" / f"garbage_{i}.jpg")
for i, img in enumerate(all_images[split:]):
    shutil.copy(img, dataset_path / "val" / "garbage" / f"garbage_{i}.jpg")

print(f"Garbage images: {len(all_images)} total")
print(f"Train: {split}, Val: {len(all_images) - split}")

In [ ]:
# Step 5: Download clean street images for 'no_garbage' class
# Using COCO or other clean images

# For simplicity, we'll create synthetic "clean" images
# In production, collect real clean street images from India

from PIL import Image
import numpy as np

def create_clean_images(output_dir, count=500):
    """Create placeholder clean images"""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    colors = [
        (100, 100, 100),  # Gray asphalt
        (34, 139, 34),    # Green grass
        (210, 180, 140),  # Tan/brown
        (176, 196, 222),  # Light steel blue
    ]
    
    for i in range(count):
        # Create base image
        img = Image.new('RGB', (640, 640), random.choice(colors))
        pixels = np.array(img)
        
        # Add texture/noise
        noise = np.random.randint(-30, 30, pixels.shape, dtype=np.int16)
        pixels = np.clip(pixels.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        
        img = Image.fromarray(pixels)
        img.save(output_dir / f"clean_{i}.jpg")
    
    print(f"Created {count} clean images in {output_dir}")

# Create clean images
create_clean_images(dataset_path / "train" / "no_garbage", count=400)
create_clean_images(dataset_path / "val" / "no_garbage", count=100)

In [ ]:
# Step 6: Train YOLOv8 Classification Model
from ultralytics import YOLO

# Load a pretrained YOLOv8 classification model
model = YOLO('yolov8n-cls.pt')  # nano model, fast
# model = YOLO('yolov8s-cls.pt')  # small model, more accurate

# Train the model
results = model.train(
    data='./garbage_dataset',
    epochs=50,
    imgsz=640,
    batch=16,
    name='garbage_classifier',
    patience=10,  # Early stopping
    device=0 if torch.cuda.is_available() else 'cpu'
)

In [ ]:
# Step 7: Evaluate model
# Load best model
best_model = YOLO('./runs/classify/garbage_classifier/weights/best.pt')

# Validate
metrics = best_model.val()
print(f"Top-1 Accuracy: {metrics.top1:.2%}")
print(f"Top-5 Accuracy: {metrics.top5:.2%}")

In [ ]:
# Step 8: Test inference
from PIL import Image
import matplotlib.pyplot as plt

# Test on a sample image
test_image = list((dataset_path / "val" / "garbage").glob("*.jpg"))[0]

results = best_model(test_image)

# Get prediction
for r in results:
    probs = r.probs
    top1_idx = probs.top1
    top1_conf = probs.top1conf
    class_name = r.names[top1_idx]
    
    print(f"Prediction: {class_name}")
    print(f"Confidence: {top1_conf:.2%}")
    
# Display image
img = Image.open(test_image)
plt.imshow(img)
plt.title(f"Prediction: {class_name} ({top1_conf:.2%})")
plt.axis('off')
plt.show()

In [ ]:
# Step 9: Export model for deployment

# Export to ONNX for faster inference
best_model.export(format='onnx')

# Export to TorchScript
best_model.export(format='torchscript')

print("Model exported!")
print("Files:")
print("  - best.pt (PyTorch)")
print("  - best.onnx (ONNX)")
print("  - best.torchscript (TorchScript)")

In [ ]:
# Step 10: Download trained model
# On Google Colab, download the model files

from google.colab import files

# Download best.pt
files.download('./runs/classify/garbage_classifier/weights/best.pt')

# Download ONNX model
files.download('./runs/classify/garbage_classifier/weights/best.onnx')

print("Models downloaded!")
print("\nNext steps:")
print("1. Place best.pt in ml-service/models/garbage_classifier.pt")
print("2. Restart the ML service")
print("3. The model will now be used for real garbage detection!")

## 🎉 Training Complete!

### Next Steps:
1. Download the `best.pt` model file
2. Place it in `ml-service/models/garbage_classifier.pt`
3. Restart the ML service
4. Your SwachhCity app will now use real ML detection!

### To Improve the Model:
1. Collect more real garbage images from Indian streets
2. Collect real "clean street" images
3. Re-train with the improved dataset
4. The more diverse your dataset, the better the model!